In [8]:
# ============================================================
#  EXERCISE: Hyperparameter Tuning with GridSearchCV
#  Dataset : Iris (3 flower species, 4 features)
#  Model   : Decision Tree Classifier
# ============================================================
#
#  YOUR TASKS are marked with  ✏️  below.
#  A fully worked solution is at the bottom of the file.
# ============================================================

from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd

# ── Load data ────────────────────────────────────────────────
iris = load_iris()
X, y = iris.data, iris.target

print("Dataset shape:", X.shape)          # (150, 4)
print("Classes:", iris.target_names)      # ['setosa', 'versicolor', 'virginica']



Dataset shape: (150, 4)
Classes: ['setosa' 'versicolor' 'virginica']


In [9]:
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()

# Convert to DataFrame
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target
df['species'] = df['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

print(df.head())
print(df.shape)       # (150, 6)
print(df.dtypes)

   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target species  
0       0  setosa  
1       0  setosa  
2       0  setosa  
3       0  setosa  
4       0  setosa  
(150, 6)
sepal length (cm)    float64
sepal width (cm)     float64
petal length (cm)    float64
petal width (cm)     float64
target                 int64
species               object
dtype: object


In [10]:
# ── Step 1: Split off a test set FIRST and lock it away ──────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTraining samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")

# Original dataset:   50 setosa | 50 versicolor | 50 virginica  (balanced, 33% each)

# Bad random split could give:
#   Train: 45 setosa | 38 versicolor | 57 virginica  ← uneven
#   Test:   5 setosa | 12 versicolor |  3 virginica  ← uneven

# with stratify=y
# Original dataset:   50 setosa | 50 versicolor | 50 virginica  (33% each)

# Train (80%):        40 setosa | 40 versicolor | 40 virginica  (33% each ✓)
# Test  (20%):        10 setosa | 10 versicolor | 10 virginica  (33% each ✓)


Training samples : 120
Test samples     : 30


In [11]:
# ── Step 2: Define the model ─────────────────────────────────
model = DecisionTreeClassifier(random_state=42)


# ── Step 3: Define the hyperparameter grid ───────────────────
# ✏️  TASK A — Fill in the param_grid below.
#     Try at least 2 hyperparameters with 3+ values each.
#     Hints:
#       'max_depth'        → controls tree depth   e.g. [2, 4, 6, None]
#       'min_samples_split' → min samples to split  e.g. [2, 5, 10]
#       'criterion'        → split quality measure  e.g. ['gini', 'entropy']

param_grid = {
    # ✏️  write your values here
    'max_depth':         [2, 4, 6, None],   # 4 values
    'min_samples_split': [2, 5, 10],         # 3 values
    'criterion':         ['gini', 'entropy'] # 2 values
}

# Total combinations = 4 × 3 × 2 = 24
# With 5-fold CV     = 24 × 5    = 120 training runs



In [12]:
# ── Step 4: Set up GridSearchCV ───────────────────────────────
# ✏️  TASK B — Create a GridSearchCV object.
#     Parameters to set:
#       estimator  → your model
#       param_grid → the grid you defined above
#       cv         → number of folds (try 5)
#       scoring    → 'accuracy'
#       verbose    → 1  (so you can see progress)

# scoring = 'accuracy'   # correct predictions / total predictions

# # Other options depending on your problem:
# scoring = 'f1'                # imbalanced classification
# scoring = 'roc_auc'           # ranking/probability problems
# scoring = 'neg_mean_squared_error'  # regression

grid_search = GridSearchCV(
    estimator  = model,
    param_grid = param_grid,
    cv         = 5,
    scoring    = 'accuracy',
    verbose    = 1
)



In [ ]:
# ── Step 5: Fit on training data ONLY ────────────────────────
# ✏️  TASK C — Fit grid_search on X_train, y_train
#     (GridSearchCV runs cross-validation internally — never pass X_test here)

grid_search.fit(X_train, y_train)
# X_train (120 samples) split into 5 folds:
# ┌────────┬────────┬────────┬────────┬────────┐
# │  F1    │  F2    │  F3    │  F4    │  F5    │
# │ 24 pts │ 24 pts │ 24 pts │ 24 pts │ 24 pts │
# └────────┴────────┴────────┴────────┴────────┘

# Combination 1: max_depth=2, min_samples_split=2
#     Run 1: Train F2+F3+F4+F5 → Test F1 → 0.92
#     Run 2: Train F1+F3+F4+F5 → Test F2 → 0.95
#     Run 3: Train F1+F2+F4+F5 → Test F3 → 0.91
#     Run 4: Train F1+F2+F3+F5 → Test F4 → 0.94
#     Run 5: Train F1+F2+F3+F4 → Test F5 → 0.93
#     Mean = 0.93 ✓

# Combination 2: max_depth=2, min_samples_split=5
#     Run 1→5 ...
#     Mean = 0.91

# ... all 12 combinations ...

# Combination 12: max_depth=None, min_samples_split=10
#     Run 1→5 ...
#     Mean = 0.88

# Winner → Combination with highest mean score
#        → Retrained on ALL 120 samples of X_train

# ── Step 6: Inspect results ───────────────────────────────────
# ✏️  TASK D — Print the best hyperparameters and best CV score.
#     Hints:
#       grid_search.best_params_   → dict of best values
#       grid_search.best_score_    → mean CV accuracy of best combo

print("\n── Grid Search Results ──────────────────────────")
# ✏️  print best_params_ and best_score_ here


# ── Step 7: Show the full results table ──────────────────────
# ✏️  TASK E — Convert grid_search.cv_results_ to a DataFrame
#     and display the top 5 rows sorted by mean_test_score descending.
#     Useful columns: 'params', 'mean_test_score', 'std_test_score'



# ── Step 8: Evaluate on the locked test set ──────────────────
# ✏️  TASK F — Use grid_search.best_estimator_ to predict on X_test
#     and print the final test accuracy.
#     Remember: this is the ONE honest evaluation — only run it here.

print("\n── Final Honest Evaluation on Test Set ─────────")
# ✏️  predict and print accuracy here


# ============================================================
#  REFLECTION QUESTIONS (think about these after running)
# ============================================================
# Q1. How many total combinations did GridSearchCV evaluate?
#     Hint: len(grid_search.cv_results_['params'])
#
# Q2. Is the test accuracy higher or lower than the best CV score?
#     Why might there be a small difference?
#
# Q3. Try adding 'criterion': ['gini', 'entropy'] to your grid.
#     How many more combinations does that add?
#
# Q4. What would happen if you called grid_search.fit(X, y)
#     instead of grid_search.fit(X_train, y_train)?
#     (This is the data leakage mistake — think about why it matters)
# ============================================================


# ============================================================
#  ✅  WORKED SOLUTION  (try it yourself first!)
# ============================================================

def worked_solution():
    from sklearn.datasets import load_iris
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.model_selection import GridSearchCV, train_test_split
    from sklearn.metrics import accuracy_score
    import pandas as pd

    # Load and split
    iris = load_iris()
    X, y = iris.data, iris.target
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Model
    model = DecisionTreeClassifier(random_state=42)

    # Hyperparameter grid — 4 × 3 × 2 = 24 combinations
    param_grid = {
        'max_depth':         [2, 4, 6, None],
        'min_samples_split': [2, 5, 10],
        'criterion':         ['gini', 'entropy'],
    }

    # GridSearchCV with 5-fold CV
    grid_search = GridSearchCV(
        estimator  = model,
        param_grid = param_grid,
        cv         = 5,
        scoring    = 'accuracy',
        verbose    = 1,
    )

    # Fit on training data only
    grid_search.fit(X_train, y_train)

    # Best results
    print("\n── Grid Search Results ──────────────────────────")
    print("Best hyperparameters :", grid_search.best_params_)
    print(f"Best CV accuracy     : {grid_search.best_score_:.4f}")

    # Full results table — top 5
    results = pd.DataFrame(grid_search.cv_results_)
    top5 = (results[['params', 'mean_test_score', 'std_test_score']]
            .sort_values('mean_test_score', ascending=False)
            .head(5)
            .reset_index(drop=True))
    print("\nTop 5 combinations:")
    print(top5.to_string())

    # Final honest evaluation
    y_pred = grid_search.best_estimator_.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)
    print("\n── Final Honest Evaluation on Test Set ─────────")
    print(f"Test accuracy : {test_acc:.4f}")

    # Reflection answer 1
    n_combos = len(grid_search.cv_results_['params'])
    print(f"\nTotal combinations evaluated : {n_combos}")
    print(f"Total CV fits run            : {n_combos} × 5 folds = {n_combos * 5}")


# ── Uncomment the line below to run the solution ─────────────
# worked_solution()

Fitting 5 folds for each of 24 candidates, totalling 120 fits

── Grid Search Results ──────────────────────────

── Final Honest Evaluation on Test Set ─────────
